# IDM-VTON Server for NexCart

**Setup:** Runtime → Change runtime type → T4 GPU → Run All Cells

After running, copy `COLAB_TRYON_URL=...` from the last cell into your `.env` file.

In [ ]:
# CELL 1: Verify GPU
import torch, subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch: {torch.__version__}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime -> Change runtime type -> T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 2: Install dependencies
#
# VERSION PINS ARE CRITICAL — DO NOT CHANGE:
#   diffusers==0.25.0   → IDM-VTON imports PositionNet which was removed in 0.26+
#   transformers==4.36.2 → paired release used during IDM-VTON development
#
# The cached_download ImportError (hub>=0.23 removed it) is fixed by the
# compatibility shim injected at the top of vton_inference.py in Cell 5.
#
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q "accelerate>=0.28.0" tqdm
!pip install -q --force-reinstall "diffusers==0.25.0" "transformers==4.36.2" "einops==0.7.0"
!pip install -q scipy opencv-python-headless Pillow
!pip install -q --no-deps fvcore iopath
!pip install -q cloudpickle omegaconf pycocotools basicsr av onnxruntime
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print()
import diffusers; print(f'diffusers {diffusers.__version__} -- must be 0.25.0')
import transformers; print(f'transformers {transformers.__version__} -- must be 4.36.2')
print('Dependency conflicts about sentence-transformers/numpy are WARNINGS only')
print('They do not affect IDM-VTON inference (server subprocess does not use them)')
# !! IMPORTANT: Runtime -> Restart session after this cell, then re-run all from Cell 1

In [ ]:
# CELL 3: Clone IDM-VTON from GitHub
import os
if not os.path.exists('/content/IDM-VTON'):
    !git clone https://github.com/yisol/IDM-VTON /content/IDM-VTON
    print('Cloned.')
else:
    print('Already cloned.')
%cd /content/IDM-VTON

In [ ]:
# CELL 4: Download model weights (~8 GB, first run takes 5-15 min)
# Downloads to /content/vton_model/ — skips on reconnect if already present
import os
MODEL_DIR = '/content/vton_model'
if os.path.exists(MODEL_DIR) and len(os.listdir(MODEL_DIR)) > 5:
    print('Model weights already downloaded, skipping.')
else:
    print('Downloading IDM-VTON weights from HuggingFace (~8 GB)...')
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='yisol/IDM-VTON',
        repo_type='space',
        local_dir=MODEL_DIR,
        ignore_patterns=[
            '*.py', 'app.py', '.gitattributes', '*.sh',
            '*.md', '__pycache__/', 'example/'
        ]
    )
    print('Download complete.')
print('Contents:', os.listdir(MODEL_DIR))

In [ ]:
# CELL 5: Write standalone inference engine (no gradio dependency)
code = (
    '# huggingface_hub compat shim — cached_download removed in hub>=0.23\nimport huggingface_hub as _hfhub\nif not hasattr(_hfhub, 'cached_download'):\n    _hfhub.cached_download = _hfhub.hf_hub_download\n'
    'import sys, os, io\n'
    'sys.path.insert(0, "/content/IDM-VTON")\n'
    'os.chdir("/content/IDM-VTON")\n'
    'import torch, logging\n'
    'import numpy as np\n'
    'from PIL import Image\n'
    'from torchvision import transforms\n'
    'from torchvision.transforms.functional import to_pil_image\n'
    'from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline\n'
    'from src.unet_hacked_garmnet import UNet2DConditionModel as UNet2DConditionModel_ref\n'
    'from src.unet_hacked_tryon import UNet2DConditionModel\n'
    'from utils_mask import get_mask_location\n'
    'import apply_net\n'
    'from preprocess.humanparsing.run_parsing import Parsing\n'
    'from preprocess.openpose.run_openpose import OpenPose\n'
    'from detectron2.data.detection_utils import convert_PIL_to_numpy\n'
    'logger = logging.getLogger("vton-inference")\n'
    'DEVICE = "cuda" if torch.cuda.is_available() else "cpu"\n'
    'BASE_PATH = "/content/vton_model"\n'
    'CKPT_DIR = "/content/vton_model/ckpt"\n'
    'tensor_tf = transforms.Compose([\n'
    '    transforms.ToTensor(),\n'
    '    transforms.Normalize([0.5], [0.5]),\n'
    '])\n'
    '_pipe = _openpose = _parsing = None\n'
    'def load_models():\n'
    '    global _pipe, _openpose, _parsing\n'
    '    if _pipe is not None: return\n'
    '    logger.info("Loading models on " + DEVICE)\n'
    '    enc = UNet2DConditionModel_ref.from_pretrained(BASE_PATH, subfolder="unet_encoder", torch_dtype=torch.float16)\n'
    '    unet = UNet2DConditionModel.from_pretrained(BASE_PATH, subfolder="unet", torch_dtype=torch.float16)\n'
    '    _pipe = TryonPipeline.from_pretrained(BASE_PATH, unet=unet, torch_dtype=torch.float16, use_safetensors=True, variant="fp16")\n'
    '    _pipe.unet_encoder = enc\n'
    '    _openpose = OpenPose(0)\n'
    '    _openpose.preprocessor.body_estimation.model.to("cpu")\n'
    '    _parsing = Parsing(0)\n'
    '    logger.info("Models loaded.")\n'
    'def run_tryon(person_img, garm_img, garment_des, auto_crop=True, denoise_steps=30, seed=42):\n'
    '    load_models()\n'
    '    garm_img = garm_img.convert("RGB").resize((768, 1024))\n'
    '    orig = person_img.convert("RGB")\n'
    '    left = top = 0; crop_size = None\n'
    '    if auto_crop:\n'
    '        w, h = orig.size\n'
    '        tw = int(min(w, h * 0.75)); th = int(min(h, w * 4/3))\n'
    '        left = (w - tw) / 2; top = (h - th) / 2\n'
    '        cropped = orig.crop((left, top, left+tw, top+th))\n'
    '        crop_size = cropped.size\n'
    '        human_img = cropped.resize((768, 1024))\n'
    '    else:\n'
    '        human_img = orig.resize((768, 1024))\n'
    '    kp = _openpose(human_img.resize((384, 512)))\n'
    '    mp, _ = _parsing(human_img.resize((384, 512)))\n'
    '    mask, _ = get_mask_location("hd", "upper_body", mp, kp)\n'
    '    mask = mask.resize((768, 1024))\n'
    '    mg = (1 - transforms.ToTensor()(mask)) * tensor_tf(human_img)\n'
    '    human_np = convert_PIL_to_numpy(human_img.resize((384, 512)), format="BGR")\n'
    '    args = apply_net.create_argument_parser().parse_args((\n'
    '        "show", "./configs/densepose_rcnn_R_50_FPN_s1x.yaml",\n'
    '        f"{CKPT_DIR}/densepose/model_final_162be9.pkl",\n'
    '        "dp_segm", "-v", "--opts", "MODEL.DEVICE", "cuda"))\n'
    '    pose_arr = args.func(args, human_np)[:,:,::-1]\n'
    '    pose_img = Image.fromarray(pose_arr).resize((768, 1024))\n'
    '    _pipe.to(DEVICE); _pipe.unet_encoder.to(DEVICE)\n'
    '    _openpose.preprocessor.body_estimation.model.to(DEVICE)\n'
    '    neg = "monochrome, lowres, bad anatomy, worst quality, low quality"\n'
    '    with torch.no_grad():\n'
    '        with torch.cuda.amp.autocast():\n'
    '            pe,npe,ppe,nppe = _pipe.encode_prompt("model is wearing "+garment_des, num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt=neg)\n'
    '            pec,_,_,_ = _pipe.encode_prompt("a photo of "+garment_des, num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt=neg)\n'
    '            pt = tensor_tf(pose_img).unsqueeze(0).to(DEVICE, torch.float16)\n'
    '            gt = tensor_tf(garm_img).unsqueeze(0).to(DEVICE, torch.float16)\n'
    '            gen = torch.Generator(DEVICE).manual_seed(seed)\n'
    '            out = _pipe(prompt_embeds=pe.to(DEVICE,torch.float16), negative_prompt_embeds=npe.to(DEVICE,torch.float16), pooled_prompt_embeds=ppe.to(DEVICE,torch.float16), negative_pooled_prompt_embeds=nppe.to(DEVICE,torch.float16), num_inference_steps=denoise_steps, generator=gen, strength=1.0, pose_img=pt, text_embeds_cloth=pec.to(DEVICE,torch.float16), cloth=gt, mask_image=mask, image=human_img, height=1024, width=768, ip_adapter_image=garm_img.resize((768,1024)), guidance_scale=2.0)[0]\n'
    '    if auto_crop and crop_size:\n'
    '        r = out[0].resize(crop_size); orig.paste(r, (int(left), int(top))); return orig\n'
    '    return out[0]\n'
)
with open('/content/vton_inference.py', 'w') as f:
    f.write(code)
print('vton_inference.py written.')

In [ ]:
# CELL 6: Write FastAPI server
srv = (
    'import sys, os, io\n'
    'sys.path.insert(0, "/content/IDM-VTON")\n'
    'os.chdir("/content/IDM-VTON")\n'
    'import torch, logging, uvicorn\n'
    'from PIL import Image\n'
    'from fastapi import FastAPI, File, UploadFile, Form, HTTPException\n'
    'from fastapi.responses import StreamingResponse\n'
    'sys.path.insert(0, "/content")\n'
    'from vton_inference import run_tryon, load_models\n'
    'logging.basicConfig(level=logging.INFO)\n'
    'logger = logging.getLogger("vton-server")\n'
    'app = FastAPI(title="IDM-VTON API")\n'
    '@app.on_event("startup")\n'
    'def startup(): load_models()\n'
    '@app.get("/health")\n'
    'def health():\n'
    '    return {"status": "ok", "model": "IDM-VTON", "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}\n'
    '@app.post("/tryon")\n'
    'async def virtual_tryon(person_image: UploadFile=File(...), garment_image: UploadFile=File(...), garment_desc: str=Form("a fashion garment"), denoise_steps: int=Form(30), seed: int=Form(42)):\n'
    '    try:\n'
    '        pb = await person_image.read(); gb = await garment_image.read()\n'
    '        p = Image.open(io.BytesIO(pb)).convert("RGB")\n'
    '        g = Image.open(io.BytesIO(gb)).convert("RGB")\n'
    '        out = run_tryon(p, g, garment_desc, auto_crop=True, denoise_steps=denoise_steps, seed=seed)\n'
    '        buf = io.BytesIO(); out.save(buf, format="JPEG", quality=90); buf.seek(0)\n'
    '        return StreamingResponse(buf, media_type="image/jpeg")\n'
    '    except Exception as e:\n'
    '        logger.error(str(e), exc_info=True)\n'
    '        raise HTTPException(status_code=500, detail=str(e))\n'
    'if __name__ == "__main__":\n'
    '    uvicorn.run(app, host="0.0.0.0", port=7860)\n'
)
with open('/content/vton_server.py', 'w') as f:
    f.write(srv)
print('vton_server.py written.')

In [ ]:
# CELL 7: Set ngrok token
# Get FREE token at: https://dashboard.ngrok.com/signup
NGROK_TOKEN = ''  # <- PASTE YOUR TOKEN HERE (inside the quotes)

if not NGROK_TOKEN:
    print('ERROR: Paste your ngrok token above first.')
    print('Get one free at https://dashboard.ngrok.com/signup')
else:
    from pyngrok import conf
    conf.get_default().auth_token = NGROK_TOKEN
    print('ngrok token configured.')

In [ ]:
# CELL 8: Start server + expose via ngrok
# Re-run ONLY this cell when Colab reconnects
import sys, os, time, subprocess, requests
from pyngrok import ngrok

ngrok.kill()

proc = subprocess.Popen(
    [sys.executable, '/content/vton_server.py'],
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT
)
print('Starting IDM-VTON server (model loading takes 60-90s on first run)...')

for i in range(90):
    time.sleep(5)
    try:
        r = requests.get('http://localhost:7860/health', timeout=3)
        if r.status_code == 200:
            print(f'\nServer ready! {r.json()}')
            break
    except:
        print('.', end='', flush=True)
else:
    print('\nServer taking too long. Last log:')
    print(open('/content/server.log').read()[-3000:])
    raise RuntimeError('Server failed to start')

public_url = ngrok.connect(7860)

print('\n' + '='*60)
print('  IDM-VTON IS LIVE ON YOUR T4 GPU!')
print('='*60)
print(f'\nPaste this into NexCart/.env:\n')
print(f'  COLAB_TRYON_URL={public_url}')
print(f'\nThen restart your Node server (npm run dev).')
print('='*60)
print('Keep this tab open — closing it kills the server.')